In [2]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing

In [3]:
class CountLeaves(MessagePassing):
    def __init__(self):
        # Note that we use the `target_to_source` flow direction because our tree has the graph laid out from root to leaves.
        super(CountLeaves, self).__init__(aggr='add', flow='target_to_source')

    def forward(self, x, edge_index):
        # Forward is how we run the whole model.
        num_nodes = x.size(0)
        return self.propagate(edge_index, size=(num_nodes, num_nodes), x=x)

    def message(self, x_j):
        # Pass the feature of the source node (x_j) as the message.
        return x_j

In [18]:
# Function to create a bifurcating tree
def create_bifurcating_tree(depth):
    edges = []
    for i in range(2**(depth+1) - 1):
        if 2 * i + 1 < 2**(depth+1) - 1:
            edges.append([i, 2 * i + 1])
            edges.append([i, 2 * i + 2])
        else:
            # add self-edges at leaf nodes
            edges.append([i, i])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

def create_caterpillar_tree(depth):
    edges = []
    for i in range(depth):
        edges.append([2*i, 2*i + 1])
        edges.append([2*i, 2*i + 2])
        # add self-edges at leaf nodes
        edges.append([2*i + 1, 2*i + 1])
    # add self-edge at leaf node
    edges.append([2*depth, 2*depth])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

depth = 3
edge_index = create_bifurcating_tree(depth)
# print("edge index:", edge_index.t())

# Define initial messages: 1 for leaves, 0 for non-leaves
num_nodes = 2**(depth+1) - 1  # Total number of nodes in the tree
x = torch.zeros(num_nodes, 1)
x[2**(depth)-1:] = 1  # Set 1 for leaf nodes

# edge index for a simple caterpillar tree
# edge_index = torch.tensor(
#     [[0,1], [0,2], [2,3], [2,4], [1,1], [3,3], [4,4]],
#     dtype=torch.long
# ).t().contiguous()
# x = torch.tensor([[0, 1, 0, 1, 1]]).t()
edge_index = create_caterpillar_tree(depth)
# print("edge index:", edge_index.t())
# Define initial messages: 1 for leaves, 0 for non-leaves
x = torch.zeros(2*depth + 1, 1)
for i in range(depth):
    x[2*i + 1] = 1
x[2*depth] = 1
# print("x:", x)

# Create data object
data = Data(x=x, edge_index=edge_index)

# Initialize and run the CountLeaves model
model = CountLeaves()
# leaf_count = model(data.x, data.edge_index)
stabilized = False
while not stabilized:
    print("Current leaf count:", data.x.t())
    leaf_count = model(data.x, data.edge_index) 
    delta = leaf_count - data.x
    if torch.count_nonzero(delta) == 0:
        print("fixed point reached")
        stabilized = True
    data.x = leaf_count
    
print("Leaf count at root node:", leaf_count[0])

Current leaf count: tensor([[0., 1., 0., 1., 0., 1., 1.]])
Current leaf count: tensor([[1., 1., 1., 1., 2., 1., 1.]])
Current leaf count: tensor([[2., 1., 3., 1., 2., 1., 1.]])
Current leaf count: tensor([[4., 1., 3., 1., 2., 1., 1.]])
fixed point reached
Leaf count at root node: tensor([4.])
